# StudyFlow: полный анализ в Python, статистика, SQL и продуктовые метрики

Полный воспроизводимый ноутбук: Python/Pandas, продуктовые метрики, статистика, A/B и SQL.


In [1]:
from pathlib import Path
import math
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'

users = pd.read_csv(RAW / 'users.csv', parse_dates=['signup_date', 'assigned_at'])
events = pd.read_csv(RAW / 'events.csv', parse_dates=['event_time'])
payments = pd.read_csv(RAW / 'payments.csv', parse_dates=['payment_date'])
support = pd.read_csv(RAW / 'support_tickets.csv', parse_dates=['created_at'])
active_users = users[users['is_test_account'] == False].copy()
users.shape, active_users.shape


((5200, 33), (5141, 33))

## 1. Качество данных


In [2]:
missing = users.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
quality_report = pd.DataFrame({'missing_rows': missing, 'missing_share': missing / len(users)})
print('user_id unique:', users['user_id'].is_unique)
quality_report


user_id unique: True


,missing_rows,missing_share
csat,4619,0.888269
age,130,0.025000
region,105,0.020192


## 2. Продуктовые KPI


In [3]:
kpi = pd.Series({
    'users': active_users['user_id'].nunique(),
    'activation_rate_7d': active_users['activated_7d'].mean(),
    'trial_start_rate_7d': active_users['trial_started'].mean(),
    'paid_conversion_14d': active_users['paid_14d'].mean(),
    'arpu_30d': active_users['revenue_30d'].mean(),
    'arppu_30d': active_users.loc[active_users['paid_14d'], 'revenue_30d'].mean(),
    'retention_30d': active_users['retained_30d'].mean(),
    'retention_60d': active_users['retained_60d'].mean(),
    'refund_rate_payers_30d': active_users.loc[active_users['paid_14d'], 'refund_30d'].mean(),
    'payment_failed_rate': active_users['payment_failed'].mean(),
    'avg_nps': active_users['nps_score'].mean(),
})
kpi


users                     5141.000000
activation_rate_7d           0.694223
trial_start_rate_7d          0.275433
paid_conversion_14d          0.223497
arpu_30d                   387.412955
arppu_30d                 1733.411662
retention_30d                0.430266
retention_60d                0.215522
refund_rate_payers_30d       0.060052
payment_failed_rate          0.052908
avg_nps                      6.950204
dtype: float64

## 3. Охват продуктовых этапов


In [4]:
funnel = pd.DataFrame([
    {'step': 'signup', 'users': active_users['user_id'].nunique()},
    {'step': 'session_start', 'users': active_users.loc[active_users['sessions_7d'] > 0, 'user_id'].nunique()},
    {'step': 'lesson_started', 'users': active_users.loc[active_users['lessons_started_7d'] > 0, 'user_id'].nunique()},
    {'step': 'activated_7d', 'users': active_users.loc[active_users['activated_7d'], 'user_id'].nunique()},
    {'step': 'paywall_seen', 'users': active_users.loc[active_users['paywall_seen'], 'user_id'].nunique()},
    {'step': 'trial_started', 'users': active_users.loc[active_users['trial_started'], 'user_id'].nunique()},
    {'step': 'paid_14d', 'users': active_users.loc[active_users['paid_14d'], 'user_id'].nunique()},
    {'step': 'retained_30d', 'users': active_users.loc[active_users['retained_30d'], 'user_id'].nunique()},
])
funnel['from_signup_rate'] = funnel['users'] / funnel.loc[0, 'users']
# Этапы не являются гарантированно вложенными, поэтому считаем только охват от регистраций.
funnel


,step,users,from_signup_rate
0,signup,5141,1.000000
1,session_start,5090,0.990080
2,lesson_started,5018,0.976075
3,activated_7d,3569,0.694223
4,paywall_seen,4995,0.971601
5,trial_started,1416,0.275433
6,paid_14d,1149,0.223497
7,retained_30d,2212,0.430266


## 4. Юнит-экономика каналов


In [5]:
channel = active_users.groupby('channel').agg(
    users=('user_id', 'nunique'),
    activated=('activated_7d', 'sum'),
    payers=('paid_14d', 'sum'),
    revenue_30d=('revenue_30d', 'sum'),
    spend=('marketing_spend_user', 'sum'),
    retained_30d=('retained_30d', 'sum'),
    refunds=('refund_30d', 'sum'),
    payment_failed=('payment_failed', 'sum'),
).reset_index()
channel['activation_rate'] = channel['activated'] / channel['users']
channel['paid_conversion'] = channel['payers'] / channel['users']
channel['arpu'] = channel['revenue_30d'] / channel['users']
channel['cac'] = channel['spend'] / channel['payers'].replace(0, np.nan)
channel['roas'] = channel['revenue_30d'] / channel['spend'].replace(0, np.nan)
channel['profit_proxy'] = channel['revenue_30d'] - channel['spend']
channel['retention_30d'] = channel['retained_30d'] / channel['users']
channel.sort_values('profit_proxy', ascending=False)


,channel,users,activated,payers,revenue_30d,spend,retained_30d,refunds,payment_failed,activation_rate,paid_conversion,arpu,cac,roas,profit_proxy,retention_30d
2,organic,1574,1041,334,558440.0,0.00,646,22,77,0.661372,0.212198,354.790343,0.000000,NaN,558440.00,0.410419
0,email,594,454,164,299060.0,44573.87,270,13,28,0.764310,0.276094,503.468013,271.791890,6.709312,254486.13,0.454545
5,referral,595,469,136,256880.0,77301.54,286,8,32,0.788235,0.228571,431.731092,568.393676,3.323090,179578.46,0.480672
1,influencer,348,244,87,150510.0,215533.65,148,3,16,0.701149,0.250000,432.500000,2477.398276,0.698313,-65023.65,0.425287
4,paid_social,900,599,191,317920.0,387367.72,368,9,66,0.665556,0.212222,353.244444,2028.103246,0.820719,-69447.72,0.408889
3,paid_search,1130,762,237,408880.0,580611.68,494,14,53,0.674336,0.209735,361.840708,2449.838312,0.704223,-171731.68,0.437168


## 5. Когорты и сегменты


In [6]:
cohort = active_users.groupby('cohort_month').agg(
    users=('user_id', 'nunique'),
    activation_rate=('activated_7d', 'mean'),
    paid_conversion=('paid_14d', 'mean'),
    retention_30d=('retained_30d', 'mean'),
    retention_60d=('retained_60d', 'mean'),
    arpu=('revenue_30d', 'mean'),
).reset_index()
segment = active_users.groupby('user_segment').agg(
    users=('user_id', 'nunique'),
    activation_rate=('activated_7d', 'mean'),
    paid_conversion=('paid_14d', 'mean'),
    arpu=('revenue_30d', 'mean'),
    retention_30d=('retained_30d', 'mean'),
    avg_lessons_completed=('lessons_completed_7d', 'mean'),
).sort_values('arpu', ascending=False)
cohort, segment


(  cohort_month  users  activation_rate  paid_conversion  retention_30d  \
 0      2026-01    873         0.687285         0.221077       0.411226   
 1      2026-02    802         0.701995         0.225686       0.421446   
 2      2026-03    844         0.680095         0.239336       0.441943   
 3      2026-04    871         0.687715         0.200918       0.411022   
 4      2026-05    903         0.700997         0.225914       0.455150   
 5      2026-06    848         0.707547         0.228774       0.439858   
 
    retention_60d        arpu  
 0       0.215349  394.455899  
 1       0.203242  381.620948  
 2       0.220379  440.379147  
 3       0.212400  335.832377  
 4       0.218162  374.551495  
 5       0.222877  399.599057  ,
                  users  activation_rate  paid_conversion        arpu  \
 user_segment                                                           
 junior_analyst    1062         0.827684         0.309793  544.905838   
 career_switcher   1631      

## 6. Статистика: доверительный интервал, t-тест, хи-квадрат и корреляция


In [7]:
paid_cr = active_users['paid_14d'].mean()
paid_se = math.sqrt(paid_cr * (1 - paid_cr) / len(active_users))
paid_ci = (paid_cr - 1.96 * paid_se, paid_cr + 1.96 * paid_se)

arpu = active_users['revenue_30d'].mean()
arpu_se = active_users['revenue_30d'].std(ddof=1) / math.sqrt(len(active_users))
arpu_ci = (arpu - 1.96 * arpu_se, arpu + 1.96 * arpu_se)

t_stat, t_p = stats.ttest_ind(
    active_users.loc[active_users['activated_7d'], 'study_minutes_7d'],
    active_users.loc[~active_users['activated_7d'], 'study_minutes_7d'],
    equal_var=False,
)
chi2, chi_p, _, _ = stats.chi2_contingency(pd.crosstab(active_users['device'], active_users['payment_failed']))
corr = active_users['lessons_completed_7d'].corr(active_users['quiz_score_after'])

pd.DataFrame([
    {'metric': 'paid_conversion_ci', 'value': paid_cr, 'ci_low': paid_ci[0], 'ci_high': paid_ci[1], 'p_value': np.nan},
    {'metric': 'arpu_ci', 'value': arpu, 'ci_low': arpu_ci[0], 'ci_high': arpu_ci[1], 'p_value': np.nan},
    {'metric': 'study_minutes_ttest', 'value': t_stat, 'ci_low': np.nan, 'ci_high': np.nan, 'p_value': t_p},
    {'metric': 'device_payment_failed_chi2', 'value': chi2, 'ci_low': np.nan, 'ci_high': np.nan, 'p_value': chi_p},
    {'metric': 'lessons_quiz_corr', 'value': corr, 'ci_low': np.nan, 'ci_high': np.nan, 'p_value': np.nan},
])


,metric,value,ci_low,ci_high,p_value
0,paid_conversion_ci,0.223497,0.212110,0.234885,NaN
1,arpu_ci,387.412955,362.027894,412.798015,NaN
2,study_minutes_ttest,61.549901,NaN,NaN,0.000000
3,device_payment_failed_chi2,5.993098,NaN,NaN,0.111946
4,lessons_quiz_corr,0.672494,NaN,NaN,NaN


## 7. A/B-тест умного онбординга


In [8]:
ab_summary = active_users.groupby('experiment_group').agg(
    users=('user_id', 'nunique'),
    activated=('activated_7d', 'sum'),
    payers=('paid_14d', 'sum'),
    arpu=('revenue_30d', 'mean'),
    retained_30d=('retained_30d', 'mean'),
    payment_failed_rate=('payment_failed', 'mean'),
    refund_rate=('refund_30d', 'mean'),
    avg_support_tickets=('support_tickets_30d', 'mean'),
    avg_nps=('nps_score', 'mean'),
)
ab_summary['activation_rate'] = ab_summary['activated'] / ab_summary['users']
ab_summary['paid_conversion'] = ab_summary['payers'] / ab_summary['users']
rows = []
for metric_name, success_col, rate_col in [('activation_rate_7d', 'activated', 'activation_rate'), ('paid_conversion_14d', 'payers', 'paid_conversion')]:
    n1 = ab_summary.loc['control', 'users']
    n2 = ab_summary.loc['smart_onboarding', 'users']
    x1 = ab_summary.loc['control', success_col]
    x2 = ab_summary.loc['smart_onboarding', success_col]
    p1 = ab_summary.loc['control', rate_col]
    p2 = ab_summary.loc['smart_onboarding', rate_col]
    pooled = (x1 + x2) / (n1 + n2)
    se = math.sqrt(pooled * (1 - pooled) * (1/n1 + 1/n2))
    z = (p2 - p1) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    rows.append({'metric': metric_name, 'control': p1, 'smart_onboarding': p2, 'uplift_pp': (p2-p1)*100, 'z_stat': z, 'p_value': p_value})
pd.DataFrame(rows), ab_summary


(                metric   control  smart_onboarding  uplift_pp     z_stat  \
 0   activation_rate_7d  0.609661          0.778555  16.889370  13.141801   
 1  paid_conversion_14d  0.186599          0.260295   7.369612   6.342060   
 
         p_value  
 0  0.000000e+00  
 1  2.267133e-10  ,
                   users  activated  payers        arpu  retained_30d  \
 experiment_group                                                       
 control            2567       1565     479  322.021815      0.384885   
 smart_onboarding   2574       2004     670  452.626263      0.475524   
 
                   payment_failed_rate  refund_rate  avg_support_tickets  \
 experiment_group                                                          
 control                      0.054149     0.012076             0.129334   
 smart_onboarding             0.051671     0.014763             0.125874   
 
                    avg_nps  activation_rate  paid_conversion  
 experiment_group                            

## 8. Запуск SQL из Python


In [9]:
conn = sqlite3.connect(PROCESSED / 'studyflow.sqlite')
# Создаём таблицу при каждом запуске: SQL-блок не зависит от заранее подготовленной базы.
users.to_sql('users', conn, if_exists='replace', index=False)
sql = '''
WITH channel_base AS (
    SELECT channel,
           COUNT(*) AS users,
           SUM(CASE WHEN paid_14d THEN 1 ELSE 0 END) AS payers,
           SUM(revenue_30d) AS revenue_30d,
           SUM(marketing_spend_user) AS spend
    FROM users
    WHERE is_test_account = 0
    GROUP BY channel
)
SELECT channel, users, payers, revenue_30d, spend,
       1.0 * payers / users AS paid_conversion,
       1.0 * spend / NULLIF(payers, 0) AS cac,
       1.0 * revenue_30d / NULLIF(spend, 0) AS roas,
       revenue_30d - spend AS profit_proxy
FROM channel_base
ORDER BY profit_proxy DESC;
'''
sql_channel = pd.read_sql_query(sql, conn)
conn.close()
sql_channel


,channel,users,payers,revenue_30d,spend,paid_conversion,cac,roas,profit_proxy
0,organic,1574,334,558440.0,0.00,0.212198,0.000000,NaN,558440.00
1,email,594,164,299060.0,44573.87,0.276094,271.791890,6.709312,254486.13
2,referral,595,136,256880.0,77301.54,0.228571,568.393676,3.323090,179578.46
3,influencer,348,87,150510.0,215533.65,0.250000,2477.398276,0.698313,-65023.65
4,paid_social,900,191,317920.0,387367.72,0.212222,2028.103246,0.820719,-69447.72
5,paid_search,1130,237,408880.0,580611.68,0.209735,2449.838312,0.704223,-171731.68


## 9. Итоговые выводы для бизнеса


In [10]:
findings = [
    'В синтетическом A/B-тесте умный онбординг повышает активацию и оплату; внедрение допустимо с контролем защитных метрик.',
    'Organic, email и referral сильнее платных каналов по приближённой прибыли.',
    'Для mobile web нужна отдельная диагностика неуспешных платежей.',
    'Этапы из пользовательских флагов нельзя считать строгой воронкой без проверки вложенности.',
]
findings

['В синтетическом A/B-тесте умный онбординг повышает активацию и оплату; внедрение допустимо с контролем защитных метрик.',
 'Organic, email и referral сильнее платных каналов по приближённой прибыли.',
 'Для mobile web нужна отдельная диагностика неуспешных платежей.',
 'Этапы из пользовательских флагов нельзя считать строгой воронкой без проверки вложенности.']